In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Chicago Inspections to Snowflake Stage") \
    .config("spark.jars.packages", "net.snowflake:snowflake-jdbc:3.13.22,net.snowflake:spark-snowflake_2.12:2.11.0-spark_3.3") \
    .getOrCreate()

# Use the provided storage account information
storage_account_name = "ayushdamg7370"
container_name = "silver"
parquet_file_name = "Chicago_All_Years_Combined.parquet"

# Set up authentication for ADLS Gen2 using the provided storage account key
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
               "xiKI4QF0gMqaJ7OZPoVraXUoU9muR4hjI3yIhpv7bVqJ0GanpCmhG423ZypkxZifOZYPzn4/EPpi+AStj4q6cw==")

# Snowflake connection parameters - replace with your actual values
snowflake_options = {
    "sfUrl": "gj27919.south-central-us.azure.snowflakecomputing.com",
    "sfUser": "Food_USER",
    "sfPassword": "snowflake123#",
    "sfDatabase": "Food_DB",
    "sfSchema": "Food_SCHEMA",
    "sfWarehouse": "Food_WH"
}

# Path to Parquet file
parquet_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{parquet_file_name}"

# Read Parquet file
df = spark.read.parquet(parquet_path)

# Display sample data (for validation)
print("Sample data from Chicago Inspections Parquet file:")
df.show(5)

# Get schema information
print("Schema of the Chicago Inspections data:")
df.printSchema()

# Name of the stage table to create in Snowflake
stage_table = "Stg_Chicago"

# Write to Snowflake stage table
df.write \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", stage_table) \
    .option("truncate_table", "true") \
    .mode("overwrite") \
    .save()

print(f"Chicago Inspections data successfully loaded to Snowflake stage table: {stage_table}")

Sample data from Chicago Inspections Parquet file:
+-------------+--------------------+--------------------+---------+-------------+---------------+--------------------+-------+-----+-------+---------------+---------------+------------------+--------------------+-----------------+------------------+--------------------+
|Inspection_ID|            DBA_Name|            AKA_Name|License_#|Facility_Type|           Risk|             Address|   City|State|    Zip|Inspection_Date|Inspection_Type|           Results|          Violations|         Latitude|         Longitude|            Location|
+-------------+--------------------+--------------------+---------+-------------+---------------+--------------------+-------+-----+-------+---------------+---------------+------------------+--------------------+-----------------+------------------+--------------------+
|      2609909|        HAPPY MARKET|        HAPPY MARKET|  2912802|Grocery Store|Risk 2 (Medium)|2334 S WENTWORTH AVE|CHICAGO|   IL|6061